# Start here

This project builds a point-in-time US equity research panel, evaluates stock-selection signals, combines five accepted ideas into one score, and produces a long-only target portfolio. It is a personal research project by Alexander Villax: the model makes the process consistent and auditable, while data exceptions, proposed trades, and the investment case for each company are still reviewed by hand.

## The notebooks, in order

| Notebook | What it does | When to use it |
|---|---|---|
| **01 — Build data** | Pulls the full historical data set and builds the research panel. | First setup, after changing data history or universe rules, or after a major source refresh. |
| **02 — Monthly update** | Refreshes live prices, rebuilds the panel, creates the target portfolio and trade list, and stops for human review. | Every monthly rebalance. This is the normal operating notebook. |
| **03 — Signal research** | Runs the alignment tests and evaluates every accepted and rejected signal block. | When researching a new signal or checking whether a signal result still reproduces. |
| **04 — Backtest** | Evaluates the composite, constructs the historical portfolio, and produces performance and attribution analysis. | When model or portfolio settings change, or when preparing a performance review. |
| **05 — Stock lookup** | Lets you enter one ticker and see current and historical signal, block, and composite rankings. | When discussing a company or comparing an analyst view with the model. |

If you are unsure where to begin, open **01** only for a new installation. For a normal month, open **02**.

The complete monthly loop is: notebook **02** produces and reviews a dated target; the separate private trading repository submits the approved trades; then its reconciliation reads actual brokerage positions and writes `data/current_portfolio.csv` back here after a second confirmation. That reconciled file is the starting point for the next month's notebook **02** run.

## First-time setup

1. **Install Python and JupyterLab.** The easiest non-terminal route is Anaconda Distribution or Miniforge with Anaconda Navigator. Use Python 3.11 or newer. Install the packages listed in `requirements.txt`. Set up one environment from `requirements.txt` rather than installing packages one by one.
2. **Clone the repository.** If you do not use a terminal, open GitHub Desktop, choose **Clone repository**, and select this project.
3. **Open JupyterLab from Anaconda Navigator.** Browse to this project folder, open `notebooks`, and start with this page. Opening JupyterLab from the project folder makes imports reliable.
4. **Confirm WRDS access.** You need permission for the CRSP and Compustat libraries used by the project. Open `config.py` in JupyterLab's text editor, find `WRDS_USERNAME`, and set it to your WRDS username if the displayed username is not yours. Never put a WRDS password in this project. WRDS will prompt securely or use its normal password file.
5. **Open notebook 01 and run one cell at a time.** On a first build, set its refresh switch to `True`. Read each confirmation before moving to the next step.
6. **Use the private trading repository after approval.** This project produces the reviewed target-portfolio CSV. The private trading repository handles brokerage connectivity and credentials; after fills, its reconciliation step reads actual positions and writes `data/current_portfolio.csv` back here only after a separate human confirmation.

## What the first run costs

The first WRDS build is the slow run. Depending on WRDS load and network speed, allow **30 minutes to two hours** for the pulls and another **10–20 minutes** for the panel build. The process downloads several million rows and creates multiple Parquet files under `data/`. Later runs reuse caches and are much faster.

Typical times on a prepared machine are: monthly update **5–20 minutes**, signal research **3–8 minutes**, the full backtest and analysis **5–15 minutes**, and the first stock lookup **about one minute**. A slow WRDS query can take longer without indicating a failure. Do not restart a cell merely because it has been busy for several minutes; look for the filled-circle or busy-kernel indicator first.

## One operating rule

**Notebooks run the workflow and show results. All actual data, signal, model, and portfolio logic lives in `src/`.** A notebook must call those functions; it must not contain a second version of the calculation.

All user-adjustable data, signal, composite, portfolio, and analysis parameters live in `config.py`. Notebook variables are visible operating controls—refresh mode, overwrite confirmation, lookup ticker, and display horizon—not model definitions. To change the sample, universe, signal window, model composition, or portfolio constraint, edit `config.py` and rerun the appropriate notebook. The README contains an “if you change X, rerun Y” table.

## Who to ask

For model, notebook, or data-check questions, contact the **repository owner, Alexander Villax**. For WRDS permissions, expired credentials, or library access, contact your institution's **WRDS account administrator** or WRDS support. For a monthly rebalance, check the notes from the previous month's update to confirm what was reviewed manually.

When asking for help, send the notebook name, the step heading, and the complete red error message. Do not send a screenshot that cuts off the first or last line.

## Troubleshooting

| Problem | What it looks like | What it means | What to do |
|---|---|---|---|
| **WRDS login fails or times out** | Password prompt repeats, authentication fails, or the connection times out. | The username, password, network, account entitlement, or WRDS service is unavailable. | Check the username shown in notebook 01 or 02. Confirm you can sign in at the WRDS website. Retry once after a few minutes. If website access or a library is denied, contact the WRDS administrator; changing model code will not fix access. |
| **A required file is missing** | `FileNotFoundError` names a file under `data/`. | A prerequisite notebook or earlier cell has not produced that file. | Read the named path. Run notebook 01 from the beginning for source or panel files. If `current_portfolio.csv` is missing after trading, run the private trading repository's reconciliation against actual positions; never create an empty substitute unless this is explicitly the first launch with `FIRST_REBALANCE = True`. |
| **The live-price staleness guard raises** | The error says the latest live price exceeds `LIVE_PRICE_REBALANCE_MAX_AGE_DAYS` or the cached panel limit. | The workflow would otherwise use abandoned provisional data. | Return to notebook 02, rerun the live-price refresh with WRDS access, confirm the latest date, then rebuild the panel. Do not increase either configured limit merely to make the error disappear. |
| **A panel check fails** | The confirmation shows a blocking failure or the panel builder stops. | A join, date, duplicate, or point-in-time invariant is broken. | Stop. Save the complete check table and error, then investigate the failing check. Warnings must be read; blocking failures must not be bypassed or silently “fixed” in a notebook. |
| **A notebook was run out of order** | A variable such as `panel`, `research`, or `result` is undefined, or a later cell refers to missing output. | The current kernel has not run the prerequisite cells. | Use **Kernel → Restart Kernel**, then **Run → Run All Cells** from the top. If a required data file is missing, run the earlier numbered notebook first. |
| **A cell is still busy** | The execution marker remains busy and no new output appears. | Large data operations can be quiet for several minutes. | Wait at least the time shown at the top of the notebook. If it exceeds that range substantially, record the last printed line before interrupting the kernel and asking for help. |
| **The kernel dies or memory is exhausted** | Jupyter reports a dead kernel or restarts unexpectedly. | The full panel and several signal blocks exceeded available memory. | Close other notebooks and applications, restart Jupyter, and run only one workflow notebook. If it repeats, use a machine with at least 16 GB of memory or ask the project owner. |

## Next step

- **New installation:** open `01_build_data.ipynb`.
- **Normal month:** open `02_monthly_update.ipynb`.
- **Approved target ready for trading:** continue in the separate private trading repository; after fills, complete its reconciliation before the next monthly update.
- **Research question:** open `03_signal_research.ipynb` or `04_backtest.ipynb`.
- **Company discussion:** open `05_stock_lookup.ipynb`.